# 03 Retrieval Test

이 노트북은 최종 FAISS 검색 품질을 확인하기 위한 노트북입니다.

목적:
- final_db 로드
- FAISS 검색 테스트
- 음식명/업종/상품군 필터 검색
- 검색 결과 이미지 시각화


In [ ]:

from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path("..").resolve()
FINAL_DB_DIR = ROOT / "data" / "final_db" / "5gb"

FAISS_PATH = FINAL_DB_DIR / "faiss.index"
EMBEDDINGS_PATH = FINAL_DB_DIR / "embeddings.npy"
MAPPING_PATH = FINAL_DB_DIR / "mapping.csv"
PROMPT_METADATA_PATH = FINAL_DB_DIR / "prompt_metadata.parquet"

FINAL_DB_DIR


In [ ]:

index = faiss.read_index(str(FAISS_PATH))
embeddings = np.load(EMBEDDINGS_PATH).astype("float32")
mapping_df = pd.read_csv(MAPPING_PATH)

if PROMPT_METADATA_PATH.exists():
    prompt_df = pd.read_parquet(PROMPT_METADATA_PATH)
else:
    prompt_df = None

print("index.ntotal:", index.ntotal)
print("embeddings:", embeddings.shape)
print("mapping_df:", mapping_df.shape)

mapping_df.head()


In [ ]:

def normalize_vectors(x):
    x = x.astype("float32")
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms


def show_search_results(result_df, image_col="final_image_path", title_cols=None, n=8, cols=4):
    if title_cols is None:
        title_cols = ["original_food_name", "business_category", "product_group"]

    sample = result_df.head(n).copy()
    rows = int(np.ceil(len(sample) / cols))

    plt.figure(figsize=(cols * 4, rows * 4))

    for i, (_, row) in enumerate(sample.iterrows(), start=1):
        image_path = Path(str(row[image_col]))

        plt.subplot(rows, cols, i)
        plt.axis("off")

        try:
            img = Image.open(image_path).convert("RGB")
            plt.imshow(img)

            title_parts = []
            for col in title_cols:
                if col in row and pd.notna(row[col]):
                    title_parts.append(str(row[col]))

            if "score" in row:
                title_parts.append(f"score={row['score']:.3f}")

            plt.title("\n".join(title_parts)[:80])

        except Exception as e:
            plt.title(f"error: {e}")

    plt.tight_layout()
    plt.show()


In [ ]:

def search_by_item_index(item_index=0, top_k=8):
    query = embeddings[item_index:item_index+1].astype("float32")
    query = normalize_vectors(query)

    scores, indices = index.search(query, top_k)

    result_df = mapping_df.iloc[indices[0]].copy()
    result_df["score"] = scores[0]

    return result_df


result_df = search_by_item_index(0, top_k=8)
display(result_df[[
    "faiss_index_id",
    "original_food_name",
    "business_category",
    "product_group",
    "score",
    "final_image_path",
]])
show_search_results(result_df, n=8, cols=4)


In [ ]:

def search_by_metadata_filter(
    query_text=None,
    business_category=None,
    product_group=None,
    food_name=None,
    top_k=8,
    candidate_limit=500,
):
    df = mapping_df.copy()

    if business_category:
        df = df[df["business_category"].astype(str) == business_category]

    if product_group:
        df = df[df["product_group"].astype(str) == product_group]

    terms = []

    if query_text:
        terms.extend(str(query_text).split())

    if food_name:
        terms.append(food_name)

    if terms:
        searchable_cols = [
            "original_food_name",
            "product_name",
            "food_code",
            "business_category",
            "product_group",
            "caption",
            "prompt_keywords",
            "text_for_embedding",
        ]

        existing_cols = [col for col in searchable_cols if col in df.columns]
        search_text = df[existing_cols].fillna("").astype(str).agg(" ".join, axis=1)

        mask = pd.Series(False, index=df.index)

        for term in terms:
            mask = mask | search_text.str.contains(term, case=False, regex=False)

        matched = df[mask]

        if len(matched) > 0:
            df = matched

    if len(df) == 0:
        return df

    candidate_ids = df["faiss_index_id"].astype(int).to_numpy()
    candidate_vectors = embeddings[candidate_ids]
    query_vector = normalize_vectors(candidate_vectors).mean(axis=0, keepdims=True)
    query_vector = normalize_vectors(query_vector)

    search_k = min(max(candidate_limit, top_k), index.ntotal)
    scores, indices = index.search(query_vector.astype("float32"), search_k)

    candidate_set = set(candidate_ids.tolist())

    rows = []

    for score, idx in zip(scores[0], indices[0]):
        if int(idx) in candidate_set:
            row = mapping_df.iloc[int(idx)].copy()
            row["score"] = float(score)
            rows.append(row)

        if len(rows) >= top_k:
            break

    if not rows:
        fallback = df.head(top_k).copy()
        fallback["score"] = 0.0
        return fallback

    return pd.DataFrame(rows)



# 테스트 1: 디저트 케이크
result_df = search_by_metadata_filter(
    query_text="딸기 케이크",
    business_category="dessert",
    product_group="cake",
    top_k=8,
)

display(result_df[[
    "original_food_name",
    "business_category",
    "product_group",
    "score",
    "final_image_path",
]])
show_search_results(result_df, n=8, cols=4)



# 테스트 2: 카페 브런치
result_df = search_by_metadata_filter(
    query_text="브런치 샌드위치",
    business_category="cafe",
    product_group="brunch",
    top_k=8,
)

display(result_df[[
    "original_food_name",
    "business_category",
    "product_group",
    "score",
    "final_image_path",
]])
show_search_results(result_df, n=8, cols=4)



# 테스트 3: 주점 해산물 안주
result_df = search_by_metadata_filter(
    query_text="회 해산물 안주",
    business_category="pub",
    product_group="seafood_side",
    top_k=8,
)

display(result_df[[
    "original_food_name",
    "business_category",
    "product_group",
    "score",
    "final_image_path",
]])
show_search_results(result_df, n=8, cols=4)



# 테스트 4: 음식점 한식
result_df = search_by_metadata_filter(
    query_text="찌개 한식",
    business_category="restaurant",
    product_group="korean_food",
    top_k=8,
)

display(result_df[[
    "original_food_name",
    "business_category",
    "product_group",
    "score",
    "final_image_path",
]])
show_search_results(result_df, n=8, cols=4)



# 카테고리별 데이터 개수 확인
business_dist = mapping_df["business_category"].value_counts().reset_index()
business_dist.columns = ["business_category", "count"]
display(business_dist)

product_dist = mapping_df["product_group"].value_counts().reset_index()
product_dist.columns = ["product_group", "count"]
display(product_dist.head(30))



# 검색 결과를 prompt_rag에 넣기 좋은 형태로 확인
if prompt_df is not None:
    display(prompt_df.head(10))
else:
    print("prompt_metadata.parquet does not exist.")
